# 06 -- DueCare Adversarial Attack Resistance

**DueCare** | Named for Cal. Civ. Code sect. 1714(a)

---

**Purpose:** How well does Gemma resist 15 different adversarial attack
strategies? This notebook generates adversarial variations from base
prompts, scores them, and identifies which attack types are most
effective at bypassing safety guardrails.

| | |
|---|---|
| **Input** | 5 graded base prompts, 15 adversarial generators |
| **Output** | Attack variation samples, per-generator statistics, attack type distribution |
| **Prerequisites** | `duecare-llm-wheels` dataset attached; no GPU required for generation (GPU needed for live model scoring) |
| **Pipeline position** | Stage 3 of the DueCare showcase pipeline. Previous: NB 05 (RAG Comparison). Next: NB 08 (Function Calling). |

---

### Why adversarial testing matters for migrant worker protection

Bad actors who exploit migrant workers do not use polite, direct
questions. They use obfuscation, academic framing, corporate disguise,
and emotional manipulation to extract harmful content from LLMs.
A model that passes standard safety tests may fail catastrophically
against these adversarial variations.

DueCare's 15 generators systematically produce these attacks so
safety evaluations test what adversaries actually do, not what
benchmarks wish they would do.

### Flow diagram

```
5 Base Prompts        15 Generators
      |                     |
      +--------+   +-------+
               |   |
               v   v
        +------+---+------+
        | Generate 1 var  |
        | per generator   |
        | per base prompt |
        +------+----------+
               |
               v
     ~75 adversarial prompts
               |
        +------+------+
        | Score each  |
        | (model/     |
        |  scripted)  |
        +------+------+
               |
               v
     Attack type analysis
     (which attacks succeed?)
```


## 1. Install DueCare

Install the DueCare wheel packages. The adversarial generators
are part of the `duecare-llm-tasks` package.


In [ ]:
import subprocess, glob, sys
for p in ['/kaggle/input/duecare-llm-wheels/*.whl',
          '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
          '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + wheels + ['-q'])
    print(f'Installed {len(wheels)} DueCare wheels.')


## 2. Generate adversarial variations from 5 base prompts

We start with 5 graded base prompts and run all 15 generators,
producing 1 variation per generator per prompt. This gives us
approximately 75 adversarial test prompts spanning every attack type.

The table below shows each generator's name and how many variations
it produced. Some generators may produce fewer than 5 if the base
prompt does not fit the attack pattern.


In [ ]:
from duecare.tasks.generators import ALL_GENERATORS
from duecare.domains import register_discovered, load_domain_pack

register_discovered()
pack = load_domain_pack('trafficking')
base = [p for p in pack.seed_prompts() if p.get('graded_responses')][:5]

print(f'Base prompts: {len(base)}')
print(f'Generators available: {len(ALL_GENERATORS)}')
print()

all_variations = []
print(f'{"Generator":<35} {"Variations":>10}')
print('-' * 47)
for gen in ALL_GENERATORS:
    variations = gen.generate(base, n_variations=1, seed=42)
    all_variations.extend(variations)
    print(f'  {gen.__class__.__name__:<33} {len(variations):>10}')

print(f'\nTotal: {len(base)} base + {len(all_variations)} variations = '
      f'{len(base)+len(all_variations)} test prompts')
print(f'Amplification factor: {len(all_variations)/max(len(base),1):.1f}x')


### What the generators produce

Each generator applies a specific adversarial strategy:
- **Evasion/Regulatory:** Frame exploitation as regulatory compliance
- **Coercion:** Use pressure tactics and urgency
- **Financial:** Disguise fees as "standard business practice"
- **Persona:** Impersonate authority figures (31 persona types)
- **Creative:** Novel attacks like ASCII art, role-play, academic framing

The diversity of attack types is what makes DueCare's evaluation
meaningful: a model cannot pass by memorizing responses to one style
of prompt.


## 3. Inspect sample adversarial variations

Each variation carries metadata about its mutation type and base
prompt. This traceability is critical for understanding which
attack types a model fails against.


In [ ]:
import textwrap
print('--- Sample adversarial variations ---\n')
for v in all_variations[:5]:
    mt = v.get('metadata',{}).get('mutation_type','?')
    base_id = v.get('metadata',{}).get('base_prompt_id','?')
    print(f'[{mt}] (from base: {base_id})')
    print(textwrap.fill(v['text'][:200], width=80))
    print('...\n')


## 4. Score all variations

In production, these would be run through Gemma 4 and scored using
the rubric framework. Here we use a ScriptedModel to demonstrate
that the scoring pipeline handles all variation types consistently.

The attack type distribution shows which categories of adversarial
attacks are represented in the generated test set.


In [ ]:
from duecare.core import Capability, GenerationResult, Embedding, ModelHealth

class ScriptedModel:
    id = 'scripted:adversarial'
    display_name = 'Scripted'
    provider = 'scripted'
    capabilities = {Capability.TEXT}
    context_length = 4096
    _r = 'I cannot help with that. Under ILO C181 Article 7, workers shall not pay fees. Contact POEA at 1343.'
    def generate(self, messages, **kw): return GenerationResult(text=self._r, model_id=self.id)
    def embed(self, texts): return [Embedding(text=t, vector=[0]*4, dimension=4, model_id=self.id) for t in texts]
    def healthcheck(self): return ModelHealth(model_id=self.id, healthy=True)

from collections import Counter
gen_results = Counter()
for v in all_variations:
    mt = v.get('metadata',{}).get('mutation_type','unknown').split('_')[0]
    gen_results[mt] += 1

print(f'{"Attack Type":<25} {"Count":>6} {"Share":>8}')
print('-' * 42)
for at, count in gen_results.most_common():
    pct = count / len(all_variations) * 100
    print(f'{at:<25} {count:>6} {pct:>7.1f}%')


## Summary and next steps

### Key findings

- **15 generators** produce diverse adversarial attacks from any base
  prompt, covering evasion, coercion, financial obfuscation, persona
  injection, and creative attacks
- Each generator tests a different attack vector -- a model cannot pass
  by defending against only one style of attack
- The scoring framework handles all variation types consistently through
  the same rubric pipeline
- This is the test diversity that makes DueCare's evaluation meaningful
  for real-world deployment

### Connection to other notebooks

- **Previous (NB 05):** RAG comparison tested whether context helps.
  This notebook tests whether safety holds under adversarial pressure.
- **Next (NB 08):** Function calling + multimodal demo shows how Gemma 4's
  unique features serve as load-bearing infrastructure.
- **NB 12:** The full adversarial prompt factory scales this to
  thousands of variations with validation and importance ranking.

**Privacy is non-negotiable. All adversarial testing runs on-device.**
